In [0]:
catalog = "flights_project"
df = spark.table(f"{catalog}.silver.flights_clean")

# Binary label: 1 if delayed, 0 otherwise (excludes cancelled/diverted — they're not "on-time vs delayed" cases)
from pyspark.sql.functions import col, when

df_model = (df
    .filter(col("flight_status").isin("ON_TIME", "DELAYED"))
    .withColumn("label", when(col("flight_status") == "DELAYED", 1.0).otherwise(0.0))
    .select("OP_UNIQUE_CARRIER", "ORIGIN", "dep_hour", "day_of_week", "is_weekend", "distance", "label")
    .na.drop()
)

display(df_model.limit(10))
print(f"Total rows: {df_model.count()}")

OP_UNIQUE_CARRIER,ORIGIN,dep_hour,day_of_week,is_weekend,distance,label
AA,LAX,7,3,false,2475.0,1.0
AA,MIA,17,3,false,1096.0,1.0
AA,PIT,12,3,false,1067.0,1.0
AA,DEN,14,3,false,1558.0,1.0
AA,ANC,6,3,false,3043.0,0.0
AA,PHX,8,3,false,2917.0,1.0
AA,PHX,15,3,false,1814.0,1.0
AA,FSD,6,3,false,737.0,0.0
AA,DTW,5,3,false,986.0,0.0
AA,DCA,14,3,false,719.0,0.0


Total rows: 2314380


In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

carrier_indexer = StringIndexer(inputCol="OP_UNIQUE_CARRIER", outputCol="carrier_idx", handleInvalid="keep")
origin_indexer = StringIndexer(inputCol="ORIGIN", outputCol="origin_idx", handleInvalid="keep")

carrier_encoder = OneHotEncoder(inputCol="carrier_idx", outputCol="carrier_vec")
origin_encoder = OneHotEncoder(inputCol="origin_idx", outputCol="origin_vec")

assembler = VectorAssembler(
    inputCols=["carrier_vec", "origin_vec", "dep_hour", "day_of_week", "is_weekend", "distance"],
    outputCol="features"
)

feature_pipeline = Pipeline(stages=[carrier_indexer, origin_indexer, carrier_encoder, origin_encoder, assembler])

In [0]:
from pyspark.ml.classification import LogisticRegression
import mlflow

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

full_pipeline = Pipeline(stages=[carrier_indexer, origin_indexer, carrier_encoder, origin_encoder, assembler, lr])

with mlflow.start_run(run_name="delay_risk_logistic_regression"):
    model = full_pipeline.fit(train_df)
    predictions = model.transform(test_df)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")

mlflow.log_metric("auc", auc)
mlflow.log_metric("accuracy", accuracy)

AUC: 0.6634
Accuracy: 0.7861


In [0]:
predictions.select("OP_UNIQUE_CARRIER", "ORIGIN", "dep_hour", "day_of_week", "label", "prediction", "probability") \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.delay_risk_predictions")

In [0]:
from pyspark.sql import Row

example_flights = spark.createDataFrame([
    Row(OP_UNIQUE_CARRIER="DL", ORIGIN="ORD", dep_hour=18, day_of_week=6, is_weekend=False, distance=733.0),   # Delta, Chicago, Fri evening
    Row(OP_UNIQUE_CARRIER="F9", ORIGIN="DEN", dep_hour=17, day_of_week=6, is_weekend=False, distance=1000.0),  # Frontier, Denver, Fri evening
    Row(OP_UNIQUE_CARRIER="HA", ORIGIN="HNL", dep_hour=9,  day_of_week=3, is_weekend=False, distance=250.0),   # Hawaiian, Honolulu, midweek morning
    Row(OP_UNIQUE_CARRIER="WN", ORIGIN="MDW", dep_hour=21, day_of_week=1, is_weekend=True,  distance=500.0),   # Southwest, Chicago Midway, Sunday night
])

scored_examples = model.transform(example_flights)
scored_examples.select("OP_UNIQUE_CARRIER", "ORIGIN", "dep_hour", "day_of_week", "prediction", "probability").show(truncate=False)


+-----------------+------+--------+-----------+----------+----------------------------------------+
|OP_UNIQUE_CARRIER|ORIGIN|dep_hour|day_of_week|prediction|probability                             |
+-----------------+------+--------+-----------+----------+----------------------------------------+
|DL               |ORD   |18      |6          |0.0       |[0.6803489150526366,0.3196510849473634] |
|F9               |DEN   |17      |6          |0.0       |[0.6441912105867551,0.3558087894132449] |
|HA               |HNL   |9       |3          |0.0       |[0.9095844686705357,0.09041553132946434]|
|WN               |MDW   |21      |1          |0.0       |[0.5571726041667059,0.4428273958332941] |
+-----------------+------+--------+-----------+----------+----------------------------------------+



In [0]:
from pyspark.sql import Row

mixed_examples = spark.createDataFrame([
    # Known low-risk (from your gold tables)
    Row(OP_UNIQUE_CARRIER="HA", ORIGIN="HNL", dep_hour=9,  day_of_week=3, is_weekend=False, distance=250.0),   # Hawaiian, calm morning
    Row(OP_UNIQUE_CARRIER="OO", ORIGIN="ATL", dep_hour=10, day_of_week=3, is_weekend=False, distance=400.0),   # SkyWest, mid-morning, midweek

    # Moderate
    Row(OP_UNIQUE_CARRIER="DL", ORIGIN="ORD", dep_hour=18, day_of_week=6, is_weekend=False, distance=733.0),   # Delta, Friday evening
    Row(OP_UNIQUE_CARRIER="AA", ORIGIN="DFW", dep_hour=17, day_of_week=6, is_weekend=False, distance=800.0),   # American, Friday rush

    # Known high-risk (from your route_risk_features table)
    Row(OP_UNIQUE_CARRIER="G4", ORIGIN="CKB", dep_hour=20, day_of_week=6, is_weekend=False, distance=250.0),   # Allegiant, CKB->SFB, notoriously bad route
    Row(OP_UNIQUE_CARRIER="G4", ORIGIN="TOL", dep_hour=21, day_of_week=5, is_weekend=False, distance=700.0),   # Allegiant, TOL 9pm — your worst airport/hour slot
    Row(OP_UNIQUE_CARRIER="B6", ORIGIN="BOS", dep_hour=20, day_of_week=6, is_weekend=False, distance=2500.0),  # JetBlue, BOS->PDX, cross-country evening
])

scored_mixed = model.transform(mixed_examples)
scored_mixed.select("OP_UNIQUE_CARRIER", "ORIGIN", "dep_hour", "day_of_week", "prediction", "probability").show(truncate=False)

+-----------------+------+--------+-----------+----------+----------------------------------------+
|OP_UNIQUE_CARRIER|ORIGIN|dep_hour|day_of_week|prediction|probability                             |
+-----------------+------+--------+-----------+----------+----------------------------------------+
|HA               |HNL   |9       |3          |0.0       |[0.9095844686705357,0.09041553132946434]|
|OO               |ATL   |10      |3          |0.0       |[0.8691996572445891,0.13080034275541086]|
|DL               |ORD   |18      |6          |0.0       |[0.6803489150526366,0.3196510849473634] |
|AA               |DFW   |17      |6          |0.0       |[0.6350405325339381,0.3649594674660619] |
|G4               |CKB   |20      |6          |1.0       |[0.33604051201323193,0.6639594879867681]|
|G4               |TOL   |21      |5          |1.0       |[0.4497452067118565,0.5502547932881434] |
|B6               |BOS   |20      |6          |0.0       |[0.582226597854087,0.41777340214591296] |
